In [1]:
import sys

sys.path.append("../..")
import torch

from interpreto.concepts.metrics import ConSim
from interpreto.model_wrapping.llm_interface import HuggingFaceLLM

In [ ]:
# Define classes
classes = ["negative", "positive"]
metric = ConSim(classes=classes)

# Sample inputs
samples = [
    "I loved the movie.",
    "It was boring and slow.",
    "Excellent acting and story.",
    "I would not recommend it.",
]

# Ground truth labels and model predictions
labels = torch.tensor([1, 0, 1, 0])
predictions = torch.tensor([1, 0, 1, 0])

# Concept interpretations
concepts_interpretation = {
    0: "positive sentiment words",
    1: "negative sentiment words",
}

# Global importances
global_importances = torch.tensor(
    [
        [-0.8, 0.7],
        [0.8, -0.7],
    ]
)

# Local importances (per sample)
local_importances = [
    torch.tensor([[-0.1, 0.7], [0.1, -0.7]]),
    torch.tensor([[0.7, -0.1], [-0.7, 0.1]]),
    torch.tensor([[-0.2, 0.8], [0.2, -0.8]]),
    torch.tensor([[0.8, -0.2], [-0.8, 0.2]]),
]

# Build prompts
system_prompt, user_prompts, model_predictions = metric.construct_prompt(
    setting=ConSim.prompt_types.E3_global_and_local_concepts_with_lp,
    interesting_samples=samples,
    corresponding_predictions=predictions,
    corresponding_labels=labels,
    nb_learning_samples=2,
    concepts_interpretation=concepts_interpretation,
    global_importances=global_importances,
    local_importances=local_importances,
)

print(f"Built {len(user_prompts)} query prompt(s).")

Built 2 query prompt(s).


In [5]:
# HuggingFaceLLM now supports both chat-template and plain CausalLM tokenizers.
# Choose any local/cached CausalLM checkpoint.

llm = HuggingFaceLLM(model="Qwen/Qwen3-0.6B", batch_size=2, device="auto")

responses = llm.batch_generate(
    system_prompt,
    user_prompts,
    max_new_tokens=40,
    do_sample=False,
)

print("Raw responses:")
for response in responses:
    print("---")
    print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Raw responses:
---
.public!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!
---
.public!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!+x!


In [4]:
score = metric.score_from_responses(responses, model_predictions)
print("ConSim score:", score)

ConSim score: 0.0
